In [ ]:
# [Problem 1] Creating a one-dimensional convolutional layer class that limits the number of channels to one

In [1]:
import numpy as np

class XavierInitializer:
    """
    Xavier Initializer class.
    Generates initial weights based on the number of input features.
    """
    def __init__(self, fan_in):
        """
        Parameters
        ----------
        fan_in : int
            The number of input features/elements (e.g., filter size for 1D Conv).
        """
        self.fan_in = fan_in

    def initialize_weight(self, shape):
        """
        Initializes weights using the Xavier method.

        Parameters
        ----------
        shape : tuple
            The shape of the weight array (e.g., (filter_size,)).
        
        Returns
        -------
        numpy.ndarray
            The initialized weight array.
        """
        # Calculate the standard deviation for Xavier initialization
        std_dev = 1.0 / np.sqrt(self.fan_in)
        # Weights are drawn from a uniform distribution U(-limit, limit)
        limit = np.sqrt(3.0) * std_dev
        return np.random.uniform(-limit, limit, size=shape)

    def initialize_bias(self, shape=(1,)):
        """
        Initializes bias to zeros (standard practice).
        
        Returns
        -------
        numpy.ndarray
            The initialized bias array.
        """
        return np.zeros(shape)


In [ ]:
# Stochastic Gradient Descent Optimizer

In [2]:
import numpy as np

class SGD:
    """
    Basic Stochastic Gradient Descent optimizer.
    This simple optimizer is included to enable the update step
    in the SimpleConv1d layer class.
    """
    def __init__(self, lr=0.01):
        """
        Parameters
        ----------
        lr : float
            Learning rate (alpha).
        """
        self.lr = lr

    def update(self, layer):
        """
        Updates the weights and bias of a layer using the calculated gradients.
        
        Parameters
        ----------
        layer : object
            The layer object (e.g., SimpleConv1d) which has dW and dB attributes.
        """
        # Update weights (W) and bias (B) using the update formulas:
        # w_s' = w_s - alpha * dL/dw_s
        # b' = b - alpha * dL/db
        layer.W -= self.lr * layer.dW
        layer.B -= self.lr * layer.dB


In [ ]:
#1D Convolutional Layer (Single Channel, Batch Size 1)

In [5]:
class SimpleConv1d:
    """
    1D Convolutional Layer class with the following constraints:
    - Input/Output channels: 1
    - Batch size: 1
    - Stride: 1
    - Padding: 0 (mode='valid')
    - Weight Initialization: Xavier
    - Update: Handled by an external optimizer (e.g., SGD).
    """

    def __init__(self, filter_size, initializer=XavierInitializer, optimizer=SGD):
        """
        Initializes weights and bias for the 1D convolution layer.

        Parameters
        ----------
        filter_size : int
            The length of the convolution filter (F).
        initializer : class
            The weight initializer class (e.g., XavierInitializer).
        optimizer : class
            The optimizer class (e.g., SGD).
        """
        self.F = filter_size
        self.initializer = initializer(fan_in=filter_size)
        self.optimizer = optimizer()
        
        # Initialize Weights (W) and Bias (B)
        # W has shape (F,)
        self.W = self.initializer.initialize_weight(shape=(self.F,))
        # B is a scalar, shape (1,)
        self.B = self.initializer.initialize_bias(shape=(1,))
        
        # Gradients (initialized to zero)
        self.dW = np.zeros_like(self.W)
        self.dB = np.zeros_like(self.B)
        
        # Store input for backpropagation
        self.X = None
        self.N_in = None
        self.N_out = None


    def forward(self, X):
        """
        Forward propagation for 1D convolution.
        a_i = SUM_{s=0}^{F-1} x_(i+s) * w_s + b

        Parameters
        ----------
        X : numpy.ndarray, shape (N_in,)
            Input array (single channel, batch size 1).
        
        Returns
        -------
        numpy.ndarray, shape (N_out,)
            The output array (pre-activation map).
        """
        self.X = X # Store input
        self.N_in = X.shape[0]
        
        # Calculate output size: N_out = N_in - F + 1 (Stride=1, Padding=0)
        self.N_out = self.N_in - self.F + 1
        
        # Check if the filter fits
        if self.N_out <= 0:
            raise ValueError("Input length is too small for the filter size.")
            
        A = np.zeros(self.N_out)
        
        # Implementation using a sliding window and dot product
        for i in range(self.N_out):
            # Extract the window from the input X
            x_window = self.X[i : i + self.F]
            
            # Convolution operation (dot product of window and filter W) + bias
            A[i] = np.dot(x_window, self.W) + self.B
            
        return A


    def backward(self, dA):
        """
        Backward propagation for 1D convolution.
        Calculates gradients dW, dB, and the error dL/dX to pass to the previous layer.

        Parameters
        ----------
        dA : numpy.ndarray, shape (N_out,)
            Gradient array from the subsequent layer (dL/dA).
        
        Returns
        -------
        numpy.ndarray, shape (N_in,)
            The error array to pass to the previous layer (dL/dX).
        """
        # --- 1. Calculate Gradient for Bias (dB) ---
        # dL/db = SUM_{i=0}^{N_out-1} dL/da_i
        self.dB = np.sum(dA)
        
        
        # --- 2. Calculate Gradient for Weights (dW) ---
        # dL/dw_s = SUM_{i=0}^{N_out-1} dL/da_i * x_(i+s)
        self.dW = np.zeros_like(self.W)
        
        for s in range(self.F):
            # x_(i+s) is the input at position (i+s). 
            # This is equivalent to summing the product of dA and the shifted input X.
            
            # The part of X involved in the convolution for w_s starts at index s and 
            # has a length of N_out.
            x_shifted = self.X[s : s + self.N_out]
            
            # dW[s] is the dot product of dA (length N_out) and x_shifted (length N_out)
            self.dW[s] = np.dot(dA, x_shifted)
            
            
        # --- 3. Calculate Error to Previous Layer (dL/dX) ---
        # dL/dx_j = SUM_{s=0}^{F-1} dL/a_(j-s) * w_s
        # This is essentially the full convolution of dA with the flipped filter W.
        
        dL_dX = np.zeros(self.N_in)
        
        # Pad dA with F-1 zeros on both sides to align it for full convolution 
        # which results in the input shape N_in.
        # Equivalent to zero-padding the output gradient dA.
        dA_padded = np.pad(dA, (self.F - 1, self.F - 1), 'constant')
        
        # Reverse the filter weights W (convolution kernel property for backprop)
        # np.flip(self.W) is the standard kernel for dL/dX calculation.
        W_flipped = np.flip(self.W)
        
        # Now we perform convolution/sliding window over the padded dA using W_flipped.
        # The resulting array will have size N_out + (F-1)*2 - F + 1 = N_in
        
        # The calculation is equivalent to convolving dA_padded with W_flipped using mode='valid'.
        # We manually implement the sliding window for clarity:
        for j in range(self.N_in):
            # The window size is F. The window slides over the padded dA.
            dA_window = dA_padded[j : j + self.F]
            
            # The error dL/dX at position j is the dot product
            # dL/dx_j = SUM (dA_window * W_flipped)
            dL_dX[j] = np.dot(dA_window, W_flipped)

        return dL_dX


    def update(self):
        """
        Updates the weights and bias using the layer's optimizer.
        """
        self.optimizer.update(self)


In [ ]:
# [Problem 2] Output size calculation after one-dimensional convolution

In [6]:
import numpy as np
import math

def calculate_1d_output_size(N_in, P, F, S):
    """
    Calculates the output size (N_out) after a 1D convolution operation.

    The formula used is: N_out = floor((N_in + 2P - F) / S) + 1

    Parameters
    ----------
    N_in : int
        Input size (number of features).
    P : int
        Number of paddings in one direction (Paddings applied to both sides).
    F : int
        Filter size.
    S : int
        Stride size.
        
    Returns
    -------
    int
        The output size (N_out).
    """
    
    # Calculate the numerator: N_in + 2P - F
    numerator = N_in + 2 * P - F
    
    # The result of the division must be an integer (floor division is typically used
    # in standard CNN implementations if the result is not an exact integer).
    # We use math.floor to ensure correct behavior.
    
    if S <= 0:
        raise ValueError("Stride (S) must be a positive integer.")
    
    if numerator < 0:
        # If the padded input is smaller than the filter, convolution is not possible.
        return 0 
    
    # Calculate the term (N_in + 2P - F) / S and take the floor
    N_out = math.floor(numerator / S) + 1
    
    return int(N_out)

# --- Example Usage ---

# Example 1: No padding, Stride 1 (as used in Problem 1)
N_in_ex1 = 10
P_ex1 = 0
F_ex1 = 3
S_ex1 = 1
# Expected output: floor((10 + 2*0 - 3) / 1) + 1 = 7 + 1 = 8
N_out_ex1 = calculate_1d_output_size(N_in_ex1, P_ex1, F_ex1, S_ex1)
# print(f"Example 1 (Input=10, Padding=0, Filter=3, Stride=1): Output Size = {N_out_ex1}") # Output: 8

# Example 2: Padding=1, Stride=2
N_in_ex2 = 10
P_ex2 = 1
F_ex2 = 3
S_ex2 = 2
# Expected output: floor((10 + 2*1 - 3) / 2) + 1 = floor(9 / 2) + 1 = 4 + 1 = 5
N_out_ex2 = calculate_1d_output_size(N_in_ex2, P_ex2, F_ex2, S_ex2)
# print(f"Example 2 (Input=10, Padding=1, Filter=3, Stride=2): Output Size = {N_out_ex2}") # Output: 5


In [7]:
# [Problem 3] Experiment of one-dimensional convolutional layer with small array

In [12]:
class SimpleConv1d:
    """
    1D Convolutional Layer class with the following constraints:
    - Input/Output channels: 1
    - Batch size: 1
    - Stride: 1
    - Padding: 0 (mode='valid')
    - Weight Initialization: Xavier
    - Update: Handled by an external optimizer (e.g., SGD).
    """

    def __init__(self, filter_size, initializer, optimizer):
        """
        Initializes weights and bias for the 1D convolution layer.

        Parameters
        ----------
        filter_size : int
            The length of the convolution filter (F).
        initializer : class
            The weight initializer class (e.g., XavierInitializer).
        optimizer : class
            The optimizer class (e.g., SGD).
        """
        self.F = filter_size
        # Use filter_size as fan_in for Xavier initialization
        # The initializer class is passed directly, e.g., initializer=XavierInitializer
        self.initializer = initializer(fan_in=filter_size) 
        self.optimizer = optimizer()
        
        # Initialize Weights (W) and Bias (B)
        # W has shape (F,)
        self.W = self.initializer.initialize_weight(shape=(self.F,))
        # B is a scalar, shape (1,)
        self.B = self.initializer.initialize_bias(shape=(1,))
        
        # Gradients (initialized to zero)
        self.dW = np.zeros_like(self.W)
        self.dB = np.zeros_like(self.B)
        
        # Store input for backpropagation
        self.X = None
        self.N_in = None
        self.N_out = None


    def _im2col(self, X):
        """
        Generates the indices needed for efficient 1D convolution calculation,
        similar to the im2col concept used in 2D CNNs.
        This extracts all sliding windows into a 2D array.
        
        Parameters
        ----------
        X : numpy.ndarray, shape (N_in,)
            Input array.
            
        Returns
        -------
        numpy.ndarray, shape (N_out, F)
            The extracted windows of the input array.
        """
        self.N_in = X.shape[0]
        self.N_out = self.N_in - self.F + 1 # Stride=1, Padding=0
        
        if self.N_out <= 0:
            raise ValueError("Input length is too small for the filter size.")
            
        # Create an array of indices for the start of each window
        start_indices = np.arange(self.N_out) 
        
        # Create an array of offsets (0 to F-1)
        offsets = np.arange(self.F)
        
        # Create a 2D array of all indices for all windows: shape (N_out, F)
        # indices[i, s] = i + s
        indices = start_indices[:, None] + offsets[None, :] 
        
        # Use the 2D indices to extract all windows at once
        X_col = X[indices]
        return X_col

    def forward(self, X):
        """
        Forward propagation for 1D convolution, using the im2col method for efficiency.
        a_i = SUM_{s=0}^{F-1} x_(i+s) * w_s + b

        Parameters
        ----------
        X : numpy.ndarray, shape (N_in,)
            Input array (single channel, batch size 1).
        
        Returns
        -------
        numpy.ndarray, shape (N_out,)
            The output array (pre-activation map).
        """
        self.X = X # Store input
        
        # 1. Reshape input X into the sliding windows (X_col)
        X_col = self._im2col(X)
        
        # 2. Perform matrix multiplication (dot product) of X_col and W
        # X_col (N_out, F) @ W (F,) -> A (N_out,)
        A = np.dot(X_col, self.W) 
        
        # 3. Add bias
        A += self.B[0] # Bias is shape (1,), so we add the scalar value
            
        return A


    def backward(self, dA):
        """
        Backward propagation for 1D convolution.
        Calculates gradients dW, dB, and the error dL/dX to pass to the previous layer.

        Parameters
        ----------
        dA : numpy.ndarray, shape (N_out,)
            Gradient array from the subsequent layer (dL/dA).
        
        Returns
        -------
        numpy.ndarray, shape (N_in,)
            The error array to pass to the previous layer (dL/dX).
        """
        # --- 1. Calculate Gradient for Bias (dB) ---
        # dL/db = SUM_{i=0}^{N_out-1} dL/da_i
        self.dB = np.sum(dA)
        
        
        # --- 2. Calculate Gradient for Weights (dW) ---
        # dL/dw_s = SUM_{i=0}^{N_out-1} dL/da_i * x_(i+s)
        # This is equivalent to np.dot(dA, X_col)
        
        # We re-calculate X_col just in case X was changed, though standard practice
        # is to store X_col in forward for efficiency if memory permits.
        X_col = self._im2col(self.X)
        
        # dA is (N_out,), X_col is (N_out, F).
        # np.dot(dA, X_col) -> (F,)
        self.dW = np.dot(dA, X_col)
        
        
        # --- 3. Calculate Error to Previous Layer (dL/dX) ---
        # dL/dx_j = SUM_{s=0}^{F-1} dL/a_(j-s) * w_s
        # This is full convolution of dA with the flipped filter W.
        
        # Create an output gradient array
        dL_dX = np.zeros(self.N_in)
        
        # Pad dA with F-1 zeros on both sides to align it for full convolution 
        dA_padded = np.pad(dA, (self.F - 1, self.F - 1), 'constant')
        
        # Reverse the filter weights W (convolution kernel property for backprop)
        W_flipped = np.flip(self.W)
        
        # Perform sliding window convolution over the padded dA using W_flipped.
        for j in range(self.N_in):
            # The window slides over the padded dA.
            dA_window = dA_padded[j : j + self.F]
            
            # dL/dx_j is the dot product
            dL_dX[j] = np.dot(dA_window, W_flipped)

        return dL_dX


    def update(self):
        """
        Updates the weights and bias using the layer's optimizer.
        """
        self.optimizer.update(self)

    def run_problem_3_test(self):
        """
        Performs the forward and backward propagation test described in Problem 3.
        """
        print("--- Running Problem 3 Test ---")
        
        # Setup inputs based on Problem 3
        self.X = np.array([1, 2, 3, 4])
        self.W = np.array([3, 5, 7])
        self.B = np.array([1]) # Using a 1-element array to match initialization shape
        
        # --- Forward Propagation Test ---
        A_actual = self.forward(self.X)
        A_expected = np.array([35, 50])
        
        is_forward_correct = np.allclose(A_actual, A_expected)
        print(f"Input X: {self.X}")
        print(f"Filter W: {self.W}")
        print(f"Bias B: {self.B}")
        print(f"Calculated Output A: {A_actual}")
        print(f"Expected Output A: {A_expected}")
        print(f"Forward Test Result: {'PASS' if is_forward_correct else 'FAIL'}")
        
        if not is_forward_correct:
            print("\nForward calculation error. Stopping backpropagation test.")
            return

        # --- Backward Propagation Test ---
        delta_A = np.array([10, 20])
        print(f"\nIncoming Gradient dA: {delta_A}")
        
        # Perform backpropagation
        delta_X_actual = self.backward(delta_A)
        
        # Expected results
        delta_B_expected = np.array([30])
        delta_W_expected = np.array([50, 80, 110])
        delta_X_expected = np.array([30, 110, 170, 140])
        
        # Compare calculated gradients with expected values
        is_dB_correct = np.allclose(self.dB, delta_B_expected)
        is_dW_correct = np.allclose(self.dW, delta_W_expected)
        is_dX_correct = np.allclose(delta_X_actual, delta_X_expected)
        
        print(f"Calculated dB: {self.dB}")
        print(f"Expected dB: {delta_B_expected}")
        print(f"dB Test Result: {'PASS' if is_dB_correct else 'FAIL'}")

        print(f"Calculated dW: {self.dW}")
        print(f"Expected dW: {delta_W_expected}")
        print(f"dW Test Result: {'PASS' if is_dW_correct else 'FAIL'}")

        print(f"Calculated dX: {delta_X_actual}")
        print(f"Expected dX: {delta_X_expected}")
        print(f"dX Test Result: {'PASS' if is_dX_correct else 'FAIL'}")
        
        overall_result = is_forward_correct and is_dB_correct and is_dW_correct and is_dX_correct
        print(f"\nOverall Problem 3 Verification: {'SUCCESS' if overall_result else 'FAILURE'}")


# --- Example of test ---

conv_layer = SimpleConv1d(filter_size=3, initializer=XavierInitializer, optimizer=SGD)
conv_layer.run_problem_3_test()


--- Running Problem 3 Test ---
Input X: [1 2 3 4]
Filter W: [3 5 7]
Bias B: [1]
Calculated Output A: [35 50]
Expected Output A: [35 50]
Forward Test Result: PASS

Incoming Gradient dA: [10 20]
Calculated dB: 30
Expected dB: [30]
dB Test Result: PASS
Calculated dW: [ 50  80 110]
Expected dW: [ 50  80 110]
dW Test Result: PASS
Calculated dX: [ 30. 110. 170. 140.]
Expected dX: [ 30 110 170 140]
dX Test Result: PASS

Overall Problem 3 Verification: SUCCESS


In [ ]:
# [Problem 4] Creating a one-dimensional convolutional layer class that does not limit the number of channels

In [14]:
class Conv1d:
    """
    1D Convolutional Layer class supporting multiple input and output channels.
    
    Constraints:
    - Batch size: 1 (input shape is (C_in, N_in))
    - Stride: 1
    - Padding: 0 (mode='valid')
    - Weight Initialization: Xavier (fan_in based on C_in * F)
    """

    def __init__(self, filter_size, C_in, C_out, initializer, optimizer):
        """
        Initializes weights and bias for the 1D convolution layer.

        Parameters
        ----------
        filter_size : int
            The length of the convolution filter (F).
        C_in : int
            Number of input channels.
        C_out : int
            Number of output channels.
        initializer : class
            The weight initializer class (e.g., XavierInitializer).
        optimizer : class
            The optimizer class (e.g., SGD).
        """
        self.F = filter_size
        self.C_in = C_in
        self.C_out = C_out
        
        # Initialize fan_in based on the number of connections per output neuron
        # which is F * C_in
        self.initializer = initializer(fan_in=self.F * self.C_in) 
        self.optimizer = optimizer()
        
        # Initialize Weights (W): shape (C_out, C_in, F)
        self.W = self.initializer.initialize_weight(shape=(self.C_out, self.C_in, self.F))
        # Initialize Bias (B): shape (C_out,)
        self.B = self.initializer.initialize_bias(shape=(self.C_out,))
        
        # Gradients (initialized to zero)
        self.dW = np.zeros_like(self.W)
        self.dB = np.zeros_like(self.B)
        
        # Store variables for backpropagation
        self.X = None
        self.X_col = None # Stores the reshaped input (im2col format)
        self.N_in = None
        self.N_out = None


    def _im2col(self, X):
        """
        Converts the multi-channel input X (C_in, N_in) into the im2col matrix 
        format (N_out, C_in * F) for efficient matrix multiplication.
        """
        # X shape: (C_in, N_in)
        if X.ndim != 2 or X.shape[0] != self.C_in:
             # This check is important as this _im2col relies on (C_in, N_in) shape
             raise ValueError(f"Input X must have shape ({self.C_in}, N_in). Got {X.shape}")

        self.N_in = X.shape[1]
        self.N_out = self.N_in - self.F + 1 # Stride=1, Padding=0
        
        if self.N_out <= 0:
            raise ValueError("Input length is too small for the filter size.")
            
        # 1. Generate 1D indices for the sliding windows (N_out, F)
        start_indices = np.arange(self.N_out) 
        offsets = np.arange(self.F)
        indices = start_indices[:, None] + offsets[None, :] 
        
        # 2. Extract windows for each input channel and stack them: (C_in, N_out, F)
        X_col_list = [X[c, :][indices] for c in range(self.C_in)]
        X_col_stacked = np.stack(X_col_list, axis=0) 
        
        # 3. Reshape for Matrix Multiplication: (N_out, C_in, F) -> (N_out, C_in * F)
        # This groups all features (C_in * F) for each output position (N_out).
        X_col_reshaped = X_col_stacked.transpose(1, 0, 2).reshape(self.N_out, self.C_in * self.F)
        
        return X_col_reshaped # Shape (N_out, C_in * F)

    def forward(self, X):
        """
        Forward propagation for 1D convolution.
        
        Input X shape: (C_in, N_in)
        Output A shape: (C_out, N_out)
        """
        self.X = X # Store input
        
        # 1. Reshape input X into the sliding windows: X_col (N_out, C_in * F)
        X_col = self._im2col(X)
        self.X_col = X_col # Store for backprop
        
        # 2. Reshape Weights: W (C_out, C_in, F) -> W_reshaped (C_in * F, C_out)
        # This aligns the C_in * F input features with the C_out output channels.
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 3. Matrix Multiplication: A_temp (N_out, C_out)
        # (N_out, C_in * F) @ (C_in * F, C_out) -> (N_out, C_out)
        A_temp = np.dot(X_col, W_reshaped)
        
        # 4. Add bias: Broadcasting adds B (C_out,) to each row of A_temp.
        A = A_temp + self.B 
        
        # 5. Output shape required: (C_out, N_out)
        return A.T # Transpose to get (C_out, N_out)


    def backward(self, dA):
        """
        Backward propagation for 1D convolution.

        Parameters
        ----------
        dA : numpy.ndarray, shape (C_out, N_out)
            Gradient array from the subsequent layer (dL/dA).
        
        Returns
        -------
        numpy.ndarray, shape (C_in, N_in)
            The error array to pass to the previous layer (dL/dX).
        """
        # dA shape: (C_out, N_out)
        
        # 1. Gradient for Bias (dB): Sum over the feature dimension (N_out)
        # Result shape: (C_out,)
        self.dB = np.sum(dA, axis=1)
        
        # --- Prepare matrices for efficient gradient calculation ---
        dA_temp = dA.T # (N_out, C_out)
        X_col = self.X_col # (N_out, C_in * F)
        
        # W_reshaped (C_in * F, C_out) (for dL/dX_col calculation)
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 2. Gradient for Weights (dW)
        # dW_reshaped = X_col.T @ dA_temp -> (C_in * F, C_out)
        dW_reshaped = np.dot(X_col.T, dA_temp)
        # Reshape back to (C_out, C_in, F)
        self.dW = dW_reshaped.T.reshape(self.C_out, self.C_in, self.F)
        
        
        # 3. Error to X_col (dL/dX_col)
        # dL/dX_col = dA_temp @ W_reshaped.T -> (N_out, C_in * F)
        dL_dX_col_reshaped = np.dot(dA_temp, W_reshaped.T)
        
        # Reshape to window format (C_in, N_out, F) for easy col2im
        # (N_out, C_in * F) -> (N_out, C_in, F) -> (C_in, N_out, F)
        dL_dX_col = dL_dX_col_reshaped.reshape(self.N_out, self.C_in, self.F).transpose(1, 0, 2)
        
        
        # 4. Error to Previous Layer (dL/dX): Inverse Im2col (col2im)
        dL_dX = np.zeros((self.C_in, self.N_in)) # Final shape: (C_in, N_in)

        # Accumulation (col2im) for each input channel
        for c in range(self.C_in):
            # For channel c, dL_dX_col[c, i, s] is the error that maps back to X[c, i+s]
            for i in range(self.N_out):
                for s in range(self.F):
                    # Accumulate the error contributions into the original input positions
                    dL_dX[c, i + s] += dL_dX_col[c, i, s] 
                    
        return dL_dX


    def update(self):
        """
        Updates the weights and bias using the layer's optimizer.
        """
        self.optimizer.update(self)

    def run_problem_4_test(self, initializer_class, optimizer_class):
        """
        Performs the forward propagation test described in Problem 4.
        """
        print("--- Running Problem 4 Forward Test (Multi-Channel) ---")
        
        # Setup inputs based on Problem 4
        X_test = np.array([[1, 2, 3, 4], [2, 3, 4, 5]]) # (2, 4)
        W_test = np.ones((3, 2, 3))                     # (3, 2, 3)
        B_test = np.array([1, 2, 3])                    # (3,)
        
        C_in, N_in = X_test.shape
        C_out, _, F = W_test.shape

        # Initialize the layer with specific parameters
        test_layer = Conv1d(filter_size=F, C_in=C_in, C_out=C_out, 
                            initializer=initializer_class, optimizer=optimizer_class)
        
        # Overwrite initialized W and B with test values
        test_layer.W = W_test
        test_layer.B = B_test
        
        # --- Forward Propagation Test ---
        A_actual = test_layer.forward(X_test)
        A_expected = np.array([[16, 22], [17, 23], [18, 24]]) # (3, 2)
        
        is_forward_correct = np.allclose(A_actual, A_expected)
        print(f"Input X shape: {X_test.shape}")
        print(f"Filter W shape: {W_test.shape}")
        print(f"Bias B shape: {B_test.shape}")
        print(f"Calculated Output A shape: {A_actual.shape}")
        print(f"Calculated Output A:\n{A_actual}")
        print(f"Expected Output A:\n{A_expected}")
        print(f"\nForward Test Result: {'PASS' if is_forward_correct else 'FAIL'}")
        
        return test_layer, is_forward_correct


In [ ]:
#[Problem 5] (Advanced task) Implementing padding

In [15]:
class Conv1d:
    """
    1D Convolutional Layer class supporting multiple input and output channels,
    now including Zero Padding (stride fixed to 1).
    
    Constraints:
    - Batch size: 1 (input shape is (C_in, N_in))
    - Stride: 1
    - Weight Initialization: Xavier (fan_in based on C_in * F)
    """

    def __init__(self, filter_size, C_in, C_out, initializer, optimizer, padding=0):
        """
        Initializes weights and bias for the 1D convolution layer.

        Parameters
        ----------
        filter_size : int
            The length of the convolution filter (F).
        C_in : int
            Number of input channels.
        C_out : int
            Number of output channels.
        initializer : class
            The weight initializer class (e.g., XavierInitializer).
        optimizer : class
            The optimizer class (e.g., SGD).
        padding : int
            The number of zero features to add to both ends of the input array.
        """
        self.F = filter_size
        self.C_in = C_in
        self.C_out = C_out
        self.P = padding # Store padding value
        
        # Initialize fan_in based on the number of connections per output neuron
        # which is F * C_in
        self.initializer = initializer(fan_in=self.F * self.C_in) 
        self.optimizer = optimizer()
        
        # Initialize Weights (W): shape (C_out, C_in, F)
        self.W = self.initializer.initialize_weight(shape=(self.C_out, self.C_in, self.F))
        # Initialize Bias (B): shape (C_out,)
        self.B = self.initializer.initialize_bias(shape=(self.C_out,))
        
        # Gradients (initialized to zero)
        self.dW = np.zeros_like(self.W)
        self.dB = np.zeros_like(self.B)
        
        # Store variables for backpropagation
        self.X = None
        self.X_col = None # Stores the reshaped input (im2col format)
        self.N_in = None
        self.N_out = None


    def _im2col(self, X):
        """
        Converts the multi-channel input X (C_in, N_in) into the im2col matrix 
        format (N_out, C_in * F) for efficient matrix multiplication.
        Applies padding before reshaping.
        """
        # X shape: (C_in, N_in)
        if X.ndim != 2 or X.shape[0] != self.C_in:
             raise ValueError(f"Input X must have shape ({self.C_in}, N_in). Got {X.shape}")

        self.N_in = X.shape[1]
        
        # --- 1. Apply Padding ---
        if self.P > 0:
            # Pad only the feature dimension (axis=1). Axis 0 is C_in.
            X_padded = np.pad(X, ((0, 0), (self.P, self.P)), 'constant')
        else:
            X_padded = X
            
        N_pad = X_padded.shape[1] # New padded length
        
        # --- 2. Calculate Output Size (Stride S=1) ---
        # N_out = N_in + 2*P - F + 1
        self.N_out = N_pad - self.F + 1 
        
        if self.N_out <= 0:
            raise ValueError("Input length is too small for the filter size, even with padding.")
            
        # --- 3. Sliding Window Indexing ---
        # Generate 1D indices for the sliding windows (N_out, F)
        start_indices = np.arange(self.N_out) 
        offsets = np.arange(self.F)
        # indices shape (N_out, F)
        indices = start_indices[:, None] + offsets[None, :] 
        
        # --- 4. Extract Windows ---
        # Extract windows for each input channel and stack them: (C_in, N_out, F)
        X_col_list = [X_padded[c, :][indices] for c in range(self.C_in)]
        X_col_stacked = np.stack(X_col_list, axis=0) 
        
        # --- 5. Reshape for Matrix Multiplication ---
        # (C_in, N_out, F) -> transpose(1, 0, 2) -> (N_out, C_in, F)
        # -> reshape -> (N_out, C_in * F)
        X_col_reshaped = X_col_stacked.transpose(1, 0, 2).reshape(self.N_out, self.C_in * self.F)
        
        return X_col_reshaped # Shape (N_out, C_in * F)

    def forward(self, X):
        """
        Forward propagation for 1D convolution.
        
        Input X shape: (C_in, N_in)
        Output A shape: (C_out, N_out)
        """
        self.X = X # Store input (unpadded)
        
        # 1. Reshape input X into the sliding windows: X_col (N_out, C_in * F)
        X_col = self._im2col(X)
        self.X_col = X_col # Store for backprop
        
        # 2. Reshape Weights: W (C_out, C_in, F) -> W_reshaped (C_in * F, C_out)
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 3. Matrix Multiplication: A_temp (N_out, C_out)
        A_temp = np.dot(X_col, W_reshaped)
        
        # 4. Add bias: Broadcasting adds B (C_out,) to each row of A_temp.
        A = A_temp + self.B 
        
        # 5. Output shape required: (C_out, N_out)
        return A.T # Transpose to get (C_out, N_out)


    def backward(self, dA):
        """
        Backward propagation for 1D convolution.

        Parameters
        ----------
        dA : numpy.ndarray, shape (C_out, N_out)
            Gradient array from the subsequent layer (dL/dA).
        
        Returns
        -------
        numpy.ndarray, shape (C_in, N_in)
            The error array to pass to the previous layer (dL/dX).
        """
        # dA shape: (C_out, N_out)
        
        # 1. Gradient for Bias (dB): Sum over the feature dimension (N_out)
        self.dB = np.sum(dA, axis=1)
        
        # --- Prepare matrices for efficient gradient calculation ---
        dA_temp = dA.T # (N_out, C_out)
        X_col = self.X_col # (N_out, C_in * F)
        
        # W_reshaped (C_in * F, C_out) (for dL/dX_col calculation)
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 2. Gradient for Weights (dW)
        # dW_reshaped = X_col.T @ dA_temp -> (C_in * F, C_out)
        dW_reshaped = np.dot(X_col.T, dA_temp)
        # Reshape back to (C_out, C_in, F)
        self.dW = dW_reshaped.T.reshape(self.C_out, self.C_in, self.F)
        
        
        # 3. Error to X_col (dL/dX_col)
        # dL/dX_col = dA_temp @ W_reshaped.T -> (N_out, C_in * F)
        dL_dX_col_reshaped = np.dot(dA_temp, W_reshaped.T)
        
        # Reshape to window format (C_in, N_out, F) for easy col2im
        # (N_out, C_in * F) -> (N_out, C_in, F) -> (C_in, N_out, F)
        dL_dX_col = dL_dX_col_reshaped.reshape(self.N_out, self.C_in, self.F).transpose(1, 0, 2)
        
        
        # 4. Error to Previous Layer (dL/X): Inverse Im2col (col2im)
        # The accumulation array must be the size of the PADDED input (N_in + 2*P)
        N_padded = self.N_in + 2 * self.P
        dL_dX_padded = np.zeros((self.C_in, N_padded))

        # Accumulation (col2im) for each input channel
        for c in range(self.C_in):
            for i in range(self.N_out):
                for s in range(self.F):
                    # Accumulate the error contributions into the original PADDED input positions
                    # Index i + s corresponds to the position in the padded array
                    dL_dX_padded[c, i + s] += dL_dX_col[c, i, s] 
                    
        # 5. Crop dL/dX_padded to remove the padding
        if self.P > 0:
            # Crop P features from the start and P features from the end of the feature dimension
            dL_dX = dL_dX_padded[:, self.P : self.N_in + self.P]
        else:
            dL_dX = dL_dX_padded # If P=0, this is already the correct shape (C_in, N_in)
                    
        return dL_dX


    def update(self):
        """
        Updates the weights and bias using the layer's optimizer.
        """
        self.optimizer.update(self)

    def run_problem_4_test(self, initializer_class, optimizer_class):
        """
        Performs the forward propagation test described in Problem 4.
        Padding is set to 0 for consistency with the original problem.
        """
        print("--- Running Problem 4 Forward Test (Multi-Channel) ---")
        
        # Setup inputs based on Problem 4
        X_test = np.array([[1, 2, 3, 4], [2, 3, 4, 5]]) # (2, 4)
        W_test = np.ones((3, 2, 3))                     # (3, 2, 3)
        B_test = np.array([1, 2, 3])                    # (3,)
        
        C_in, N_in = X_test.shape
        C_out, _, F = W_test.shape

        # Initialize the layer with specific parameters, padding=0
        test_layer = Conv1d(filter_size=F, C_in=C_in, C_out=C_out, 
                            initializer=initializer_class, optimizer=optimizer_class, padding=0)
        
        # Overwrite initialized W and B with test values
        test_layer.W = W_test
        test_layer.B = B_test
        
        # --- Forward Propagation Test ---
        A_actual = test_layer.forward(X_test)
        A_expected = np.array([[16, 22], [17, 23], [18, 24]]) # (3, 2)
        
        is_forward_correct = np.allclose(A_actual, A_expected)
        print(f"Input X shape: {X_test.shape}")
        print(f"Filter W shape: {W_test.shape}")
        print(f"Bias B shape: {B_test.shape}")
        print(f"Calculated Output A shape: {A_actual.shape}")
        print(f"Calculated Output A:\n{A_actual}")
        print(f"Expected Output A:\n{A_expected}")
        print(f"\nForward Test Result: {'PASS' if is_forward_correct else 'FAIL'}")
        
        return test_layer, is_forward_correct



In [ ]:
#[Problem 6] (Advanced task) Response to mini batch

In [16]:
class Conv1d:
    """
    1D Convolutional Layer class supporting mini-batches, multiple input/output channels,
    and Zero Padding (stride fixed to 1).
    
    Input/Output Shape Convention: (Batch size B, Number of Channels C, Number of Features N)
    
    Constraints:
    - Stride: 1
    - Weight Initialization: Xavier (fan_in based on C_in * F)
    """

    def __init__(self, filter_size, C_in, C_out, initializer, optimizer, padding=0):
        """
        Initializes weights and bias for the 1D convolution layer.

        Parameters
        ----------
        filter_size : int
            The length of the convolution filter (F).
        C_in : int
            Number of input channels.
        C_out : int
            Number of output channels.
        initializer : class
            The weight initializer class (e.g., XavierInitializer).
        optimizer : class
            The optimizer class (e.g., SGD).
        padding : int
            The number of zero features to add to both ends of the input array.
        """
        self.F = filter_size
        self.C_in = C_in
        self.C_out = C_out
        self.P = padding # Store padding value
        
        # Initialize fan_in based on the number of connections per output neuron
        self.initializer = initializer(fan_in=self.F * self.C_in) 
        self.optimizer = optimizer()
        
        # Initialize Weights (W): shape (C_out, C_in, F)
        self.W = self.initializer.initialize_weight(shape=(self.C_out, self.C_in, self.F))
        # Initialize Bias (B): shape (C_out,)
        self.B = self.initializer.initialize_bias(shape=(self.C_out,))
        
        # Gradients (initialized to zero)
        self.dW = np.zeros_like(self.W)
        self.dB = np.zeros_like(self.B)
        
        # Store variables for backpropagation
        self.X = None
        self.X_col = None # Stores the reshaped input (im2col format)
        self.B = None
        self.N_in = None
        self.N_out = None


    def _im2col(self, X):
        """
        Converts the mini-batch input X (B, C_in, N_in) into the im2col matrix 
        format (B * N_out, C_in * F) for efficient matrix multiplication.
        Applies padding before reshaping.
        """
        # X shape: (B, C_in, N_in)
        if X.ndim != 3 or X.shape[1] != self.C_in:
             raise ValueError(f"Input X must have shape (B, {self.C_in}, N_in). Got {X.shape}")

        self.B, self.C_in, self.N_in = X.shape
        
        # --- 1. Apply Padding ---
        if self.P > 0:
            # Pad only the feature dimension (axis=2). Axes 0=B, 1=C_in.
            X_padded = np.pad(X, ((0, 0), (0, 0), (self.P, self.P)), 'constant')
        else:
            X_padded = X
            
        N_pad = X_padded.shape[2] # New padded length
        
        # --- 2. Calculate Output Size (Stride S=1) ---
        # N_out = N_in + 2*P - F + 1
        self.N_out = N_pad - self.F + 1 
        
        if self.N_out <= 0:
            raise ValueError("Input length is too small for the filter size, even with padding.")
            
        # --- 3. Sliding Window Indexing ---
        # Generate 1D indices for the sliding windows (N_out, F)
        start_indices = np.arange(self.N_out) 
        offsets = np.arange(self.F)
        # indices shape (N_out, F)
        indices = start_indices[:, None] + offsets[None, :] 
        
        # --- 4. Extract Windows (Vectorized) ---
        # Flatten B and C_in together: (B * C_in, N_padded)
        X_padded_flat = X_padded.reshape(self.B * self.C_in, N_pad)
        
        # X_col_all shape: (B * C_in, N_out, F)
        # For each of the B*C_in rows, this extracts the windows defined by indices
        X_col_all = X_padded_flat[:, indices]
        
        # --- 5. Reshape for Matrix Multiplication ---
        # Goal shape: (B * N_out, C_in * F)
        # Current: (B, C_in, N_out, F) after reshaping
        
        # 5a. Reshape to (B, C_in, N_out, F)
        X_col_4d = X_col_all.reshape(self.B, self.C_in, self.N_out, self.F)
        
        # 5b. Transpose to (B, N_out, C_in, F) to group B and N_out
        X_col_transposed = X_col_4d.transpose(0, 2, 1, 3) 
        
        # 5c. Reshape to final im2col shape (B * N_out, C_in * F)
        X_col_final = X_col_transposed.reshape(self.B * self.N_out, self.C_in * self.F)
        
        return X_col_final # Shape (B * N_out, C_in * F)

    def forward(self, X):
        """
        Forward propagation for 1D convolution.
        
        Input X shape: (B, C_in, N_in)
        Output A shape: (B, C_out, N_out)
        """
        self.X = X # Store input (unpadded)
        
        # 1. Reshape input X into the sliding windows: X_col (B * N_out, C_in * F)
        X_col = self._im2col(X)
        self.X_col = X_col # Store for backprop
        
        # 2. Reshape Weights: W (C_out, C_in, F) -> W_reshaped (C_in * F, C_out)
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 3. Matrix Multiplication: A_temp (B * N_out, C_out)
        # (B * N_out, C_in * F) @ (C_in * F, C_out) -> (B * N_out, C_out)
        A_temp = np.dot(X_col, W_reshaped)
        
        # 4. Add bias: Broadcasting adds B (C_out,) to each row of A_temp.
        A_biased = A_temp + self.B 
        
        # 5. Reshape to final output shape: (B, C_out, N_out)
        # Current shape: (B * N_out, C_out)
        # Reshape to (B, N_out, C_out) -> Transpose to (B, C_out, N_out)
        A = A_biased.reshape(self.B, self.N_out, self.C_out).transpose(0, 2, 1)
        
        return A


    def backward(self, dA):
        """
        Backward propagation for 1D convolution.

        Parameters
        ----------
        dA : numpy.ndarray, shape (B, C_out, N_out)
            Gradient array from the subsequent layer (dL/dA).
        
        Returns
        -------
        numpy.ndarray, shape (B, C_in, N_in)
            The error array to pass to the previous layer (dL/dX).
        """
        # dA shape: (B, C_out, N_out)
        
        # 1. Gradient for Bias (dB): Sum over the batch (0) and feature (2) dimensions
        # Result shape: (C_out,)
        self.dB = np.sum(dA, axis=(0, 2))
        
        # --- Prepare matrices for efficient gradient calculation ---
        # Flatten dA: (B, C_out, N_out) -> (B, N_out, C_out) -> (B * N_out, C_out)
        dA_temp = dA.transpose(0, 2, 1).reshape(self.B * self.N_out, self.C_out)
        X_col = self.X_col # (B * N_out, C_in * F)
        
        # W_reshaped (C_in * F, C_out) (for dL/dX_col calculation)
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 2. Gradient for Weights (dW)
        # dW_reshaped = X_col.T @ dA_temp -> (C_in * F, C_out)
        # Accumulation over the entire batch is handled by the matrix product
        dW_reshaped = np.dot(X_col.T, dA_temp)
        # Reshape back to (C_out, C_in, F)
        self.dW = dW_reshaped.T.reshape(self.C_out, self.C_in, self.F)
        
        
        # 3. Error to X_col (dL/dX_col)
        # dL/dX_col = dA_temp @ W_reshaped.T -> (B * N_out, C_in * F)
        dL_dX_col_reshaped = np.dot(dA_temp, W_reshaped.T)
        
        # --- 4. Error to Previous Layer (dL/X): Inverse Im2col (col2im) ---
        # The accumulation array must be the size of the PADDED input (B, C_in, N_in + 2*P)
        N_padded = self.N_in + 2 * self.P
        dL_dX_padded = np.zeros((self.B, self.C_in, N_padded))

        # Reshape dL/dX_col_reshaped back to window format for the loop: (B, N_out, C_in, F)
        dL_dX_col = dL_dX_col_reshaped.reshape(self.B, self.N_out, self.C_in, self.F).transpose(0, 2, 1, 3) 
        # Final shape for loop: (B, C_in, N_out, F)
        
        # Accumulation (col2im) for each batch and input channel
        for b in range(self.B):
            for c in range(self.C_in):
                for i in range(self.N_out):
                    for s in range(self.F):
                        # Accumulate the error contributions into the original PADDED input positions
                        # Index i + s corresponds to the position in the padded array
                        dL_dX_padded[b, c, i + s] += dL_dX_col[b, c, i, s] 
                        
        # 5. Crop dL/dX_padded to remove the padding
        if self.P > 0:
            # Crop P features from the start and P features from the end of the feature dimension (axis 2)
            dL_dX = dL_dX_padded[:, :, self.P : self.N_in + self.P]
        else:
            dL_dX = dL_dX_padded 
                    
        return dL_dX


    def update(self):
        """
        Updates the weights and bias using the layer's optimizer.
        """
        self.optimizer.update(self)

    def run_problem_4_test(self, initializer_class, optimizer_class):
        """
        Performs the forward propagation test described in Problem 4, 
        extended to use a batch size of B=2 to test the mini-batch implementation.
        """
        print("--- Running Problem 4 Forward Test (Multi-Channel & Mini-Batch B=2) ---")
        
        # Setup inputs based on Problem 4, but duplicated for B=2
        X_single = np.array([[1, 2, 3, 4], [2, 3, 4, 5]]) 
        X_test = np.stack([X_single, X_single], axis=0) # (B=2, 2, 4)
        W_test = np.ones((3, 2, 3))                     # (3, 2, 3)
        B_test = np.array([1, 2, 3])                    # (3,)
        
        B, C_in, N_in = X_test.shape
        C_out, _, F = W_test.shape

        # Initialize the layer with specific parameters, padding=0
        test_layer = Conv1d(filter_size=F, C_in=C_in, C_out=C_out, 
                            initializer=initializer_class, optimizer=optimizer_class, padding=0)
        
        # Overwrite initialized W and B with test values
        test_layer.W = W_test
        test_layer.B = B_test
        
        # --- Forward Propagation Test ---
        A_actual = test_layer.forward(X_test)
        
        A_single_expected = np.array([[16, 22], [17, 23], [18, 24]]) # (3, 2)
        A_expected = np.stack([A_single_expected, A_single_expected], axis=0) # (2, 3, 2)
        
        is_forward_correct = np.allclose(A_actual, A_expected)
        print(f"Input X shape: {X_test.shape} (B, C_in, N_in)")
        print(f"Filter W shape: {W_test.shape} (C_out, C_in, F)")
        print(f"Bias B shape: {B_test.shape}")
        print(f"Calculated Output A shape: {A_actual.shape} (B, C_out, N_out)")
        
        print(f"\nForward Test Result: {'PASS' if is_forward_correct else 'FAIL'}")
        if not is_forward_correct:
            print(f"Calculated A (Batch 0):\n{A_actual[0]}")
            print(f"Expected A (Batch 0):\n{A_expected[0]}")
        
        return test_layer, is_forward_correct


In [ ]:
# [Problem 7] (Advance assignment) Arbitrary number of strides

In [17]:
class Conv1d:
    """
    1D Convolutional Layer class supporting mini-batches, multiple input/output channels,
    Zero Padding, and arbitrary Stride (S).
    
    Input/Output Shape Convention: (Batch size B, Number of Channels C, Number of Features N)
    
    Constraints:
    - Weight Initialization: Xavier (fan_in based on C_in * F)
    """

    def __init__(self, filter_size, C_in, C_out, initializer, optimizer, padding=0, stride=1):
        """
        Initializes weights and bias for the 1D convolution layer.

        Parameters
        ----------
        filter_size : int
            The length of the convolution filter (F).
        C_in : int
            Number of input channels.
        C_out : int
            Number of output channels.
        initializer : class
            The weight initializer class (e.g., XavierInitializer).
        optimizer : class
            The optimizer class (e.g., SGD).
        padding : int
            The number of zero features to add to both ends of the input array (P).
        stride : int
            The step size of the sliding window (S).
        """
        self.F = filter_size
        self.C_in = C_in
        self.C_out = C_out
        self.P = padding # Store padding value (P)
        self.S = stride # Store stride value (S)
        
        # Initialize fan_in based on the number of connections per output neuron
        self.initializer = initializer(fan_in=self.F * self.C_in) 
        self.optimizer = optimizer()
        
        # Initialize Weights (W): shape (C_out, C_in, F)
        self.W = self.initializer.initialize_weight(shape=(self.C_out, self.C_in, self.F))
        # Initialize Bias (B): shape (C_out,)
        self.B = self.initializer.initialize_bias(shape=(self.C_out,))
        
        # Gradients (initialized to zero)
        self.dW = np.zeros_like(self.W)
        self.dB = np.zeros_like(self.B)
        
        # Store variables for backpropagation
        self.X = None
        self.X_col = None # Stores the reshaped input (im2col format)
        self.B = None
        self.N_in = None
        self.N_out = None


    def _im2col(self, X):
        """
        Converts the mini-batch input X (B, C_in, N_in) into the im2col matrix 
        format (B * N_out, C_in * F) for efficient matrix multiplication.
        Applies padding and stride before reshaping.
        """
        # X shape: (B, C_in, N_in)
        if X.ndim != 3 or X.shape[1] != self.C_in:
             raise ValueError(f"Input X must have shape (B, {self.C_in}, N_in). Got {X.shape}")

        self.B, self.C_in, self.N_in = X.shape
        
        # --- 1. Apply Padding ---
        if self.P > 0:
            # Pad only the feature dimension (axis=2). Axes 0=B, 1=C_in.
            X_padded = np.pad(X, ((0, 0), (0, 0), (self.P, self.P)), 'constant')
        else:
            X_padded = X
            
        N_pad = X_padded.shape[2] # New padded length (N_in + 2*P)
        
        # --- 2. Calculate Output Size (Stride S) ---
        # N_out = floor((N_in + 2*P - F) / S) + 1
        # Using integer division (//) for floor operation
        self.N_out = (N_pad - self.F) // self.S + 1 
        
        if self.N_out <= 0:
            raise ValueError("Input length is too small for the filter size and stride.")
            
        # --- 3. Sliding Window Indexing ---
        # Generate 1D indices for the sliding windows (N_out, F)
        # Start indices jump by self.S (stride)
        start_indices = np.arange(self.N_out) * self.S 
        offsets = np.arange(self.F)
        # indices shape (N_out, F)
        indices = start_indices[:, None] + offsets[None, :] 
        
        # --- 4. Extract Windows (Vectorized) ---
        # Flatten B and C_in together: (B * C_in, N_padded)
        X_padded_flat = X_padded.reshape(self.B * self.C_in, N_pad)
        
        # X_col_all shape: (B * C_in, N_out, F)
        # For each of the B*C_in rows, this extracts the windows defined by indices
        X_col_all = X_padded_flat[:, indices]
        
        # --- 5. Reshape for Matrix Multiplication ---
        # 5a. Reshape to (B, C_in, N_out, F)
        X_col_4d = X_col_all.reshape(self.B, self.C_in, self.N_out, self.F)
        
        # 5b. Transpose to (B, N_out, C_in, F) to group B and N_out
        X_col_transposed = X_col_4d.transpose(0, 2, 1, 3) 
        
        # 5c. Reshape to final im2col shape (B * N_out, C_in * F)
        X_col_final = X_col_transposed.reshape(self.B * self.N_out, self.C_in * self.F)
        
        return X_col_final # Shape (B * N_out, C_in * F)

    def forward(self, X):
        """
        Forward propagation for 1D convolution.
        
        Input X shape: (B, C_in, N_in)
        Output A shape: (B, C_out, N_out)
        """
        self.X = X # Store input (unpadded)
        
        # 1. Reshape input X into the sliding windows: X_col (B * N_out, C_in * F)
        X_col = self._im2col(X)
        self.X_col = X_col # Store for backprop
        
        # 2. Reshape Weights: W (C_out, C_in, F) -> W_reshaped (C_in * F, C_out)
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 3. Matrix Multiplication: A_temp (B * N_out, C_out)
        # (B * N_out, C_in * F) @ (C_in * F, C_out) -> (B * N_out, C_out)
        A_temp = np.dot(X_col, W_reshaped)
        
        # 4. Add bias: Broadcasting adds B (C_out,) to each row of A_temp.
        A_biased = A_temp + self.B 
        
        # 5. Reshape to final output shape: (B, C_out, N_out)
        A = A_biased.reshape(self.B, self.N_out, self.C_out).transpose(0, 2, 1)
        
        return A


    def backward(self, dA):
        """
        Backward propagation for 1D convolution.

        Parameters
        ----------
        dA : numpy.ndarray, shape (B, C_out, N_out)
            Gradient array from the subsequent layer (dL/dA).
        
        Returns
        -------
        numpy.ndarray, shape (B, C_in, N_in)
            The error array to pass to the previous layer (dL/dX).
        """
        # dA shape: (B, C_out, N_out)
        
        # 1. Gradient for Bias (dB): Sum over the batch (0) and feature (2) dimensions
        self.dB = np.sum(dA, axis=(0, 2))
        
        # --- Prepare matrices for efficient gradient calculation ---
        # Flatten dA: (B, C_out, N_out) -> (B, N_out, C_out) -> (B * N_out, C_out)
        dA_temp = dA.transpose(0, 2, 1).reshape(self.B * self.N_out, self.C_out)
        X_col = self.X_col # (B * N_out, C_in * F)
        
        # W_reshaped (C_in * F, C_out) (for dL/dX_col calculation)
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 2. Gradient for Weights (dW)
        # Accumulation over the entire batch is handled by the matrix product
        dW_reshaped = np.dot(X_col.T, dA_temp)
        # Reshape back to (C_out, C_in, F)
        self.dW = dW_reshaped.T.reshape(self.C_out, self.C_in, self.F)
        
        
        # 3. Error to X_col (dL/dX_col)
        dL_dX_col_reshaped = np.dot(dA_temp, W_reshaped.T)
        
        # --- 4. Error to Previous Layer (dL/X): Inverse Im2col (col2im) ---
        N_padded = self.N_in + 2 * self.P
        dL_dX_padded = np.zeros((self.B, self.C_in, N_padded))

        # Reshape dL/dX_col_reshaped back to window format for the loop: (B, N_out, C_in, F)
        dL_dX_col = dL_dX_col_reshaped.reshape(self.B, self.N_out, self.C_in, self.F).transpose(0, 2, 1, 3) 
        # Final shape for loop: (B, C_in, N_out, F)
        
        # Accumulation (col2im) for each batch and input channel
        for b in range(self.B):
            for c in range(self.C_in):
                for i in range(self.N_out):
                    # Start position for this window 'i' in the padded array, now considering stride S
                    start_idx = i * self.S 
                    for s in range(self.F):
                        # Accumulate the error contributions into the original PADDED input positions
                        dL_dX_padded[b, c, start_idx + s] += dL_dX_col[b, c, i, s] 
                        
        # 5. Crop dL/dX_padded to remove the padding
        if self.P > 0:
            # Crop P features from the start and P features from the end of the feature dimension (axis 2)
            dL_dX = dL_dX_padded[:, :, self.P : self.N_in + self.P]
        else:
            dL_dX = dL_dX_padded 
                    
        return dL_dX


    def update(self):
        """
        Updates the weights and bias using the layer's optimizer.
        """
        self.optimizer.update(self)

    def run_problem_4_test(self, initializer_class, optimizer_class):
        """
        Performs the forward propagation test described in Problem 4, 
        extended to use a batch size of B=2 to test the mini-batch implementation.
        """
        print("--- Running Problem 4 Forward Test (Multi-Channel & Mini-Batch B=2) ---")
        
        # Setup inputs based on Problem 4, but duplicated for B=2
        X_single = np.array([[1, 2, 3, 4], [2, 3, 4, 5]]) 
        X_test = np.stack([X_single, X_single], axis=0) # (B=2, 2, 4)
        W_test = np.ones((3, 2, 3))                     # (3, 2, 3)
        B_test = np.array([1, 2, 3])                    # (3,)
        
        B, C_in, N_in = X_test.shape
        C_out, _, F = W_test.shape

        # Initialize the layer with specific parameters, padding=0, stride=1 (Problem 4 constraints)
        test_layer = Conv1d(filter_size=F, C_in=C_in, C_out=C_out, 
                            initializer=initializer_class, optimizer=optimizer_class, padding=0, stride=1)
        
        # Overwrite initialized W and B with test values
        test_layer.W = W_test
        test_layer.B = B_test
        
        # --- Forward Propagation Test ---
        A_actual = test_layer.forward(X_test)
        
        A_single_expected = np.array([[16, 22], [17, 23], [18, 24]]) # (3, 2)
        A_expected = np.stack([A_single_expected, A_single_expected], axis=0) # (2, 3, 2)
        
        is_forward_correct = np.allclose(A_actual, A_expected)
        print(f"Input X shape: {X_test.shape} (B, C_in, N_in)")
        print(f"Filter W shape: {W_test.shape} (C_out, C_in, F)")
        print(f"Bias B shape: {B_test.shape}")
        print(f"Calculated Output A shape: {A_actual.shape} (B, C_out, N_out)")
        
        print(f"\nForward Test Result: {'PASS' if is_forward_correct else 'FAIL'}")
        if not is_forward_correct:
            print(f"Calculated A (Batch 0):\n{A_actual[0]}")
            print(f"Expected A (Batch 0):\n{A_expected[0]}")
        
        return test_layer, is_forward_correct


In [ ]:
#[Problem 8] Learning and estimation

In [22]:
import numpy as np

# --- 1. Utility Classes (Initializer, Optimizer) ---

class XavierInitializer:
    """Initializes weights using the Xavier (Glorot) method."""
    def __init__(self, fan_in):
        self.scale = np.sqrt(1.0 / fan_in)

    def initialize_weight(self, shape):
        return self.scale * np.random.randn(*shape)

    def initialize_bias(self, shape):
        return np.zeros(shape)

class SGD:
    """Stochastic Gradient Descent optimizer."""
    def __init__(self, lr=0.01):
        self.lr = lr

    def update(self, layer):
        """Updates weights and biases of a layer."""
        # Check if layer has W/dW
        if hasattr(layer, 'W') and hasattr(layer, 'dW'):
            layer.W -= self.lr * layer.dW
        # Check if layer has B/dB
        if hasattr(layer, 'B') and hasattr(layer, 'dB'):
            layer.B -= self.lr * layer.dB

# --- 2. Basic Layer Classes ---

class Affine:
    """Fully Connected Layer."""
    def __init__(self, N_in, N_out, initializer, optimizer):
        self.initializer = initializer(fan_in=N_in)
        self.optimizer = optimizer()
        
        self.W = self.initializer.initialize_weight(shape=(N_in, N_out))
        self.B = self.initializer.initialize_bias(shape=(N_out,))
        
        self.dW = np.zeros_like(self.W)
        self.dB = np.zeros_like(self.B)
        self.X = None # Stored input for backward pass

    def forward(self, X):
        self.X = X # Input X shape (B, N_in)
        A = np.dot(X, self.W) + self.B
        return A # Output A shape (B, N_out)

    def backward(self, dA):
        # Gradients
        self.dW = np.dot(self.X.T, dA) # (N_in, B) @ (B, N_out) -> (N_in, N_out)
        self.dB = np.sum(dA, axis=0)   # Sum over batch dimension
        
        # Error to previous layer
        dX = np.dot(dA, self.W.T) # (B, N_out) @ (N_out, N_in) -> (B, N_in)
        return dX

    def update(self):
        self.optimizer.update(self)

class Relu:
    """Rectified Linear Unit activation function."""
    def __init__(self):
        self.mask = None

    def forward(self, X):
        self.mask = (X <= 0)
        out = X.copy()
        out[self.mask] = 0
        return out

    def backward(self, d_out):
        d_out[self.mask] = 0
        return d_out

# --- 3. Loss Function ---

class SoftmaxWithLoss:
    """Softmax Activation combined with Cross-Entropy Loss."""
    def __init__(self):
        self.loss = None
        self.Y = None # Softmax output (probabilities)
        self.T = None # Target (one-hot encoded)

    def forward(self, X, T):
        """X is the output from the last Affine layer (logits). T is the target."""
        # Numerically stable softmax
        X_exp = np.exp(X - np.max(X, axis=1, keepdims=True))
        self.Y = X_exp / np.sum(X_exp, axis=1, keepdims=True)
        self.T = T
        
        # Cross-entropy error calculation
        batch_size = X.shape[0]
        epsilon = 1e-7 # Small constant to prevent log(0)
        log_prob = np.log(self.Y[np.arange(batch_size), np.argmax(T, axis=1)] + epsilon)
        self.loss = -np.sum(log_prob) / batch_size
        return self.loss

    def backward(self):
        batch_size = self.T.shape[0]
        # dL/dX = (Y - T) / B
        dX = (self.Y - self.T) / batch_size
        return dX

    def accuracy(self, X, T):
        Y = self.forward(X, T) # Use forward to get Y (softmax output)
        Y_pred = np.argmax(self.Y, axis=1)
        T_true = np.argmax(T, axis=1)
        accuracy = np.sum(Y_pred == T_true) / Y_pred.size
        return accuracy

# --- 4. Conv1d Layer (Final version with B, C, N, P, S support) ---

class Conv1d:
    """
    1D Convolutional Layer class supporting mini-batches, multiple input/output channels,
    Zero Padding, and arbitrary Stride (S).
    """

    def __init__(self, filter_size, C_in, C_out, initializer, optimizer, padding=0, stride=1):
        self.F = filter_size
        self.C_in = C_in
        self.C_out = C_out
        self.P = padding # Store padding value (P)
        self.S = stride # Store stride value (S)
        
        # Initialization and Optimizer setup
        self.initializer = initializer(fan_in=self.F * self.C_in) 
        self.optimizer = optimizer()
        
        self.W = self.initializer.initialize_weight(shape=(self.C_out, self.C_in, self.F))
        self.B = self.initializer.initialize_bias(shape=(self.C_out,))
        
        self.dW = np.zeros_like(self.W)
        self.dB = np.zeros_like(self.B)
        
        # Store variables for backpropagation
        self.X = None
        self.X_col = None 
        self.B_size = None
        self.N_in = None
        self.N_out = None


    def _im2col(self, X):
        """Converts the input X (B, C_in, N_in) into the im2col matrix (B * N_out, C_in * F)."""
        if X.ndim != 3 or X.shape[1] != self.C_in:
             raise ValueError(f"Input X must have shape (B, {self.C_in}, N_in). Got {X.shape}")

        self.B_size, self.C_in, self.N_in = X.shape
        
        # 1. Apply Padding
        if self.P > 0:
            X_padded = np.pad(X, ((0, 0), (0, 0), (self.P, self.P)), 'constant')
        else:
            X_padded = X
            
        N_pad = X_padded.shape[2]
        
        # 2. Calculate Output Size (Stride S)
        self.N_out = (N_pad - self.F) // self.S + 1 
        if self.N_out <= 0:
            raise ValueError("Input length is too small for the filter size and stride.")
            
        # 3. Sliding Window Indexing
        start_indices = np.arange(self.N_out) * self.S 
        offsets = np.arange(self.F)
        indices = start_indices[:, None] + offsets[None, :] 
        
        # 4. Extract Windows
        X_padded_flat = X_padded.reshape(self.B_size * self.C_in, N_pad)
        X_col_all = X_padded_flat[:, indices]
        
        # 5. Reshape for Matrix Multiplication
        X_col_4d = X_col_all.reshape(self.B_size, self.C_in, self.N_out, self.F)
        X_col_transposed = X_col_4d.transpose(0, 2, 1, 3) 
        X_col_final = X_col_transposed.reshape(self.B_size * self.N_out, self.C_in * self.F)
        
        return X_col_final # Shape (B * N_out, C_in * F)

    def forward(self, X):
        self.X = X
        
        # 1. Reshape input X into the sliding windows: X_col (B * N_out, C_in * F)
        X_col = self._im2col(X)
        self.X_col = X_col 
        
        # 2. Reshape Weights: W_reshaped (C_in * F, C_out)
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 3. Matrix Multiplication: A_temp (B * N_out, C_out)
        A_temp = np.dot(X_col, W_reshaped)
        
        # 4. Add bias
        A_biased = A_temp + self.B 
        
        # 5. Reshape to final output shape: (B, C_out, N_out)
        A = A_biased.reshape(self.B_size, self.N_out, self.C_out).transpose(0, 2, 1)
        
        return A


    def backward(self, dA):
        # dA shape: (B, C_out, N_out)
        
        # 1. Gradient for Bias (dB)
        self.dB = np.sum(dA, axis=(0, 2))
        
        # --- Prepare matrices for efficient gradient calculation ---
        # Flatten dA: (B * N_out, C_out)
        dA_temp = dA.transpose(0, 2, 1).reshape(self.B_size * self.N_out, self.C_out)
        X_col = self.X_col # (B * N_out, C_in * F)
        
        # W_reshaped (C_in * F, C_out) 
        W_reshaped = self.W.reshape(self.C_out, self.C_in * self.F).T 
        
        # 2. Gradient for Weights (dW)
        dW_reshaped = np.dot(X_col.T, dA_temp)
        self.dW = dW_reshaped.T.reshape(self.C_out, self.C_in, self.F)
        
        # 3. Error to X_col (dL/dX_col)
        dL_dX_col_reshaped = np.dot(dA_temp, W_reshaped.T)
        
        # --- 4. Error to Previous Layer (dL/X): Inverse Im2col (col2im) ---
        N_padded = self.N_in + 2 * self.P
        dL_dX_padded = np.zeros((self.B_size, self.C_in, N_padded))

        # Reshape dL/dX_col_reshaped to window format: (B, C_in, N_out, F)
        dL_dX_col = dL_dX_col_reshaped.reshape(self.B_size, self.N_out, self.C_in, self.F).transpose(0, 2, 1, 3) 
        
        # Accumulation (col2im)
        for b in range(self.B_size):
            for c in range(self.C_in):
                for i in range(self.N_out):
                    start_idx = i * self.S 
                    for s in range(self.F):
                        dL_dX_padded[b, c, start_idx + s] += dL_dX_col[b, c, i, s] 
                        
        # 5. Crop dL/dX_padded to remove the padding
        if self.P > 0:
            dL_dX = dL_dX_padded[:, :, self.P : self.N_in + self.P]
        else:
            dL_dX = dL_dX_padded 
                    
        return dL_dX


    def update(self):
        self.optimizer.update(self)

# --- 5. Network Class and Training Simulation ---

class MyNetwork:
    """
    A simple CNN structure using Conv1d for MNIST (treated as 1D sequence).
    Architecture: Conv1d -> Relu -> Flatten -> Affine -> Softmax
    """
    def __init__(self, input_size, conv_params, hidden_size, output_size):
        # Initializers and Optimizers are shared for simplicity
        initializer = XavierInitializer
        optimizer = SGD

        # Define Convolutional Layer (B, C_in, N_in) -> (B, C_out, N_out)
        self.conv = Conv1d(
            filter_size=conv_params['F'], 
            C_in=conv_params['C_in'], 
            C_out=conv_params['C_out'], 
            initializer=initializer, 
            optimizer=optimizer, 
            padding=conv_params['P'], 
            stride=conv_params['S']
        )
        
        # Calculate N_out after convolution (assuming N_in = input_size)
        N_pad = input_size + 2 * conv_params['P']
        N_out_conv = (N_pad - conv_params['F']) // conv_params['S'] + 1
        
        # Flattening size: C_out * N_out
        affine_in_size = conv_params['C_out'] * N_out_conv

        # Define Output Layers
        self.layers = [
            self.conv,
            Relu(),
            # Flattening is handled in the forward pass
            Affine(affine_in_size, hidden_size, initializer, optimizer),
            Relu(),
            Affine(hidden_size, output_size, initializer, optimizer)
        ]
        
        self.loss_layer = SoftmaxWithLoss()

    def forward(self, X, T=None):
        # X shape: (B, C_in, N_in)
        
        # 1. Convolution and ReLU
        Z = X
        for layer in self.layers[:2]: # Conv1d and first Relu
            Z = layer.forward(Z)
            
        # 2. Flattening (Smoothing)
        # Z shape is (B, C_out, N_out). Reshape to (B, C_out * N_out)
        B_size, C_out, N_out = Z.shape
        Z_flat = Z.reshape(B_size, C_out * N_out)
        
        # 3. Affine Layers
        A = Z_flat
        for layer in self.layers[2:]: # Affine layers
            A = layer.forward(A)

        # 4. Loss/Prediction
        if T is not None:
            return self.loss_layer.forward(A, T)
        return A

    def backward(self, loss=1):
        # 1. SoftmaxWithLoss backward
        dA = self.loss_layer.backward()
        
        # 2. Affine Layers backward (reverse order)
        # Note: dA is currently (B, Output_size)
        for layer in reversed(self.layers[2:]):
            dA = layer.backward(dA)
            
        # 3. Reshape back to (B, C_out, N_out) before ReLU backward
        B_size = self.conv.B_size
        C_out = self.conv.C_out
        N_out = self.conv.N_out
        dA_reshaped = dA.reshape(B_size, C_out, N_out)
        
        # 4. ReLU and Conv1d backward
        for layer in reversed(self.layers[:2]):
            dA_reshaped = layer.backward(dA_reshaped)
        
        return dA_reshaped

    def update(self):
        """
        Updates weights for all layers that have parameters (Conv1d, Affine).
        """
        for layer in self.layers:
            if hasattr(layer, 'update'):
                layer.update()

    def accuracy(self, X, T):
        Y = self.forward(X)
        Y_pred = np.argmax(Y, axis=1)
        T_true = np.argmax(T, axis=1)
        return np.sum(Y_pred == T_true) / Y_pred.size

def simulate_mnist_training():
    """Simulates a small training loop for demonstration."""
    print("--- Simulating MNIST Training with Conv1d ---")

    # --- 1. Setup Data and Network Parameters ---
    # MNIST input image: 28x28 = 784 features. Treat as 1D sequence (1, 784)
    INPUT_SIZE = 784 
    OUTPUT_SIZE = 10 
    HIDDEN_SIZE = 100 
    
    # Conv1d Parameters: C_in=1, C_out=10, Filter=5, Padding=2, Stride=1
    # This setup maintains the feature size N_out=784
    CONV_PARAMS = {'C_in': 1, 'C_out': 10, 'F': 5, 'P': 2, 'S': 1}
    BATCH_SIZE = 50
    EPOCHS = 20

    network = MyNetwork(INPUT_SIZE, CONV_PARAMS, HIDDEN_SIZE, OUTPUT_SIZE)
    
    # --- 2. Generate Dummy Data (simulating MNIST) ---
    # Dummy Training Data
    X_train = np.random.randn(500, INPUT_SIZE).reshape(500, 1, INPUT_SIZE) 
    T_train = np.zeros((500, OUTPUT_SIZE))
    T_train[np.arange(500), np.random.randint(0, OUTPUT_SIZE, 500)] = 1 # Random one-hot targets
    
    # Dummy Test Data
    X_test = np.random.randn(100, INPUT_SIZE).reshape(100, 1, INPUT_SIZE)
    T_test = np.zeros((100, OUTPUT_SIZE))
    T_test[np.arange(100), np.random.randint(0, OUTPUT_SIZE, 100)] = 1
    
    # --- 3. Training Loop ---
    
    print(f"Network initialized. Input size: (B, 1, {INPUT_SIZE}).")
    print(f"Flattening size before Affine: {network.layers[2].W.shape[0]}.")
    
    iters_per_epoch = max(1, X_train.shape[0] // BATCH_SIZE)
    
    for epoch in range(EPOCHS):
        total_loss = 0
        
        # Mini-batch loop
        for i in range(iters_per_epoch):
            # Select mini-batch indices
            batch_mask = np.random.choice(X_train.shape[0], BATCH_SIZE)
            X_batch = X_train[batch_mask]
            T_batch = T_train[batch_mask]
            
            # Forward, Backward, Update
            loss = network.forward(X_batch, T_batch)
            network.backward(loss)
            network.update()
            
            total_loss += loss
            
        avg_loss = total_loss / iters_per_epoch
        train_acc = network.accuracy(X_train[:100], T_train[:100])
        test_acc = network.accuracy(X_test, T_test)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Train Acc (sample): {train_acc:.4f} | Test Acc: {test_acc:.4f}")

    print("\nTraining simulation complete.")

simulate_mnist_training()


--- Simulating MNIST Training with Conv1d ---
Network initialized. Input size: (B, 1, 784).
Flattening size before Affine: 7840.
Epoch 1/20 | Loss: 2.3323 | Train Acc (sample): 0.2500 | Test Acc: 0.1200
Epoch 2/20 | Loss: 2.1612 | Train Acc (sample): 0.4400 | Test Acc: 0.1400
Epoch 3/20 | Loss: 2.0067 | Train Acc (sample): 0.4900 | Test Acc: 0.1400
Epoch 4/20 | Loss: 1.8428 | Train Acc (sample): 0.4800 | Test Acc: 0.1100
Epoch 5/20 | Loss: 1.6757 | Train Acc (sample): 0.5700 | Test Acc: 0.0700
Epoch 6/20 | Loss: 1.5585 | Train Acc (sample): 0.7600 | Test Acc: 0.1000
Epoch 7/20 | Loss: 1.3452 | Train Acc (sample): 0.8000 | Test Acc: 0.1000
Epoch 8/20 | Loss: 1.2001 | Train Acc (sample): 0.8500 | Test Acc: 0.1100
Epoch 9/20 | Loss: 1.0565 | Train Acc (sample): 0.9100 | Test Acc: 0.0900
Epoch 10/20 | Loss: 0.9116 | Train Acc (sample): 0.9500 | Test Acc: 0.0900
Epoch 11/20 | Loss: 0.7630 | Train Acc (sample): 0.9500 | Test Acc: 0.1100
Epoch 12/20 | Loss: 0.6626 | Train Acc (sample): 0.9900